# Offline SageMaker local mode — XGBoost pipeline

A single-`TrainingStep` SageMaker pipeline executed **locally** against moto via `sagemaker-local`. Pipeline steps run synchronously in local containers.

Dataset: `wine` (multiclass classification), loaded inside the container from scikit-learn.

In [ ]:
import os
from dataclasses import replace

from sagemaker.estimator import Estimator
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import TrainingStep
from sagemaker_local.config import config_from_env
from sagemaker_local.session import make_local_pipeline_session

cfg = replace(
    config_from_env(),
    bucket="sagemaker-xgboost",
    image_tag="sagemaker-xgboost:train",
)
boto_session, pipeline_session = make_local_pipeline_session(cfg)

## Define the pipeline

The estimator's `fit()` is captured as a `TrainingStep` request; nothing runs until the pipeline is started.

In [ ]:
est = Estimator(
    entry_point="train.py",
    source_dir=os.path.join(
        os.environ.get("SAGEMAKER_LOCAL_REPO_PATH", "/workspace"),
        "projects",
        "sagemaker_xgboost",
        "src",
        "sagemaker_xgboost",
    ),
    image_uri=cfg.image_tag,
    role=cfg.role_arn,
    instance_type="local",
    instance_count=1,
    sagemaker_session=pipeline_session,
    output_path=f"s3://{cfg.bucket}/models",
    hyperparameters={"dataset": "wine"},
)
train_step = TrainingStep(name="train-wine", estimator=est)
pipeline = Pipeline(
    name="xgboost-local-pipeline",
    steps=[train_step],
    sagemaker_session=pipeline_session,
)

## Start it

`create()` registers the pipeline with the local client; `start()` executes every step synchronously in local containers.

In [ ]:
pipeline.create(role_arn=cfg.role_arn)
execution = pipeline.start()
print("pipeline started under name", pipeline.name)